In [1]:
# IMPORT LIBRARY
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import MinMaxScaler

In [2]:
# MENDEFINISIKAN VARIABEL INPUT DAN TARGET
df = pd.read_csv("Data_Model_LSTM.csv")
df["QUOTE_DATE"] = pd.to_datetime(df["QUOTE_DATE"])
df = df.sort_values(["ID_OPTION", "QUOTE_DATE"]).reset_index(drop=True)

features = [
    "UNDERLYING_LAST",
    "STRIKE",
    "VOLATILITAS_DINAMIS",
    "r_DINAMIS",
    "DTE"
]
target = "MID_PRICE"
window = 5

In [3]:
# PEMBAGIAN DATA PELATIHAN, VALIDASI, DAN PENGUJIAN
splits = [
    ("2022-04-01","2022-11-30","2022-12-01","2022-12-30","2023-01-03","2023-01-31"),
    ("2022-05-02","2022-12-30","2023-01-03","2023-01-31","2023-02-01","2023-02-28"),
    ("2022-06-01","2023-01-31","2023-02-01","2023-02-28","2023-03-01","2023-03-31"),
    ("2022-07-01","2023-02-28","2023-03-01","2023-03-31","2023-04-03","2023-04-28"),
    ("2022-08-01","2023-03-31","2023-04-03","2023-04-28","2023-05-01","2023-05-31"),
    ("2022-09-01","2023-04-28","2023-05-01","2023-05-31","2023-06-01","2023-06-30"),
    ("2022-10-03","2023-05-31","2023-06-01","2023-06-30","2023-07-03","2023-07-31"),
    ("2022-11-01","2023-06-30","2023-07-03","2023-07-31","2023-08-01","2023-08-31"),
    ("2022-12-01","2023-07-31","2023-08-01","2023-08-31","2023-09-01","2023-09-29"),
    ("2023-01-03","2023-08-31","2023-09-01","2023-09-29","2023-10-02","2023-10-31"),
    ("2023-02-01","2023-09-29","2023-10-02","2023-10-31","2023-11-01","2023-11-30"),
    ("2023-03-01","2023-10-25","2023-11-01","2023-11-30","2023-12-01","2023-12-29")
]

In [4]:
# MEMBUAT SEQUENCES
def build_all_sequences(df, features, target, window=5):
    X_all, Y_all, dates_all, contract_ids_all = [], [], [], []

    df = df.sort_values(["ID_OPTION", "QUOTE_DATE"]).reset_index(drop=True)

    for opt_id, g in df.groupby("ID_OPTION"):
        g = g.sort_values("QUOTE_DATE").reset_index(drop=True)

        if len(g) < window:
            continue

        X_vals = g[features].values
        Y_vals = g[target].values
        date_vals = g["QUOTE_DATE"].values

        for i in range(window - 1, len(g)):
            X_seq = X_vals[i - window + 1 : i + 1]   
            y_t   = Y_vals[i]                        
            d_t   = date_vals[i]                     

            X_all.append(X_seq)
            Y_all.append(y_t)
            dates_all.append(d_t)
            contract_ids_all.append(opt_id)

    return (
        np.array(X_all, dtype=np.float32),
        np.array(Y_all, dtype=np.float32),
        pd.to_datetime(dates_all),
        np.array(contract_ids_all)
    )

In [5]:
# FUNGSI SCALING
def scale_3d(X, scaler):
    X_2d = X.reshape(-1, X.shape[-1])
    X_scaled = scaler.transform(X_2d)
    return X_scaled.reshape(X.shape)

In [6]:
# MEMBUAT SELURUH SEQUENCES SEKALI
X_all, Y_all, date_all, contract_all = build_all_sequences(df, features, target, window)

In [7]:
# PENSKALAAN DAN TRANSFORMASI DATA
for split_idx, (train_start, train_end, val_start, val_end, test_start, test_end) in enumerate(splits, start=1):

    train_start = pd.to_datetime(train_start)
    train_end   = pd.to_datetime(train_end)
    val_start   = pd.to_datetime(val_start)
    val_end     = pd.to_datetime(val_end)
    test_start  = pd.to_datetime(test_start)
    test_end    = pd.to_datetime(test_end)

    train_mask = (date_all >= train_start) & (date_all <= train_end)
    val_mask   = (date_all >= val_start)   & (date_all <= val_end)
    test_mask  = (date_all >= test_start)  & (date_all <= test_end)

    X_train, Y_train = X_all[train_mask], Y_all[train_mask]
    X_val, Y_val     = X_all[val_mask], Y_all[val_mask]
    X_test, Y_test   = X_all[test_mask], Y_all[test_mask]

    date_train, date_val, date_test = date_all[train_mask], date_all[val_mask], date_all[test_mask]
    contract_train = contract_all[train_mask]
    contract_val   = contract_all[val_mask]
    contract_test  = contract_all[test_mask]

    scaler_X = MinMaxScaler()
    scaler_X.fit(X_train.reshape(-1, X_train.shape[-1]))

    X_train_scaled = scale_3d(X_train, scaler_X)
    X_val_scaled   = scale_3d(X_val, scaler_X) if len(X_val) > 0 else X_val
    X_test_scaled  = scale_3d(X_test, scaler_X) if len(X_test) > 0 else X_test

    scaler_Y = MinMaxScaler()
    Y_train_scaled = scaler_Y.fit_transform(Y_train.reshape(-1, 1))
    Y_val_scaled   = scaler_Y.transform(Y_val.reshape(-1, 1)) if len(Y_val) > 0 else Y_val.reshape(-1, 1)
    Y_test_scaled  = scaler_Y.transform(Y_test.reshape(-1, 1)) if len(Y_test) > 0 else Y_test.reshape(-1, 1)

    np.savez_compressed(
        f"lstm_dataset_split_{split_idx}.npz",
        X_train=X_train_scaled, Y_train=Y_train_scaled,
        X_val=X_val_scaled, Y_val=Y_val_scaled,
        X_test=X_test_scaled, Y_test=Y_test_scaled,
        contract_train=contract_train,
        contract_val=contract_val,
        contract_test=contract_test,
        date_train=date_train.astype("datetime64[ns]"),
        date_val=date_val.astype("datetime64[ns]"),
        date_test=date_test.astype("datetime64[ns]")
    )

    joblib.dump(scaler_X, f"scaler_X_split_{split_idx}.pkl")
    joblib.dump(scaler_Y, f"scaler_Y_split_{split_idx}.pkl")